# 06 · Decision & action extraction

Three combined signals — never invention:

1. **DistilBERT probabilities** (when `scripts/train_classifier.py` has produced a checkpoint)
2. **Deterministic cue rules** (`decided / agreed / approved / I will / by Friday / …`)
3. **spaCy** for PERSON entities and DATE/TIME deadlines

Labels: `0 = other, 1 = decision, 2 = action`. Training labels are **rule-based weak
labels**, documented as weak (see `src/classifier.py`), with meeting-level splits to
prevent leakage. Owner/deadline strategy: explicit → `Not specified`, never guessed.

In [ ]:
print('[1/4] Cue rules and weak labels (no model needed)')
import sys; sys.path.insert(0, '.')
from src.extraction import rule_decision_hits, rule_action_hits, weak_label
for s in ['We decided to go with option B.',
          'I will send the report by Friday.',
          'The weather is lovely today.']:
    print(f'{s!r:60s} -> weak_label={weak_label(s)} '
          f'decision_cues={rule_decision_hits(s)} action_cues={rule_action_hits(s)}')

In [ ]:
print('[2/4] Optional: fine-tune DistilBERT on weak labels (needs QMSum train split)')
from pathlib import Path
from src.config import load_config, resolve_path
cfg = load_config()
ckpt = resolve_path(cfg, 'models') / 'classifier' / 'config.json'
print('checkpoint exists:', ckpt.exists())
if not ckpt.exists():
    print('Run: !python scripts/train_classifier.py --train-meetings 40')
    print('(skipping training in this cell so the notebook stays runnable)')

In [ ]:
print('[3/4] Analyse the synthetic sample with rules (+ model if checkpoint exists)')
from src.classifier import SentenceClassifier, sentences_from_meetings
from src.data_loader import load_transcript_file
from src.preprocessing import Turn, clean_text, turn_sentences
from src.extraction import extract_decisions, extract_actions, extract_followups
from src.utils import select_models

m = load_transcript_file('data/samples/sample_meeting.txt')
turns = [Turn(t['speaker'], clean_text(t['text']), t.get('start'), t.get('end'))
         for t in m['turns'] if t['text']]
sents = turn_sentences(turns, m['meeting_id'])
clf = SentenceClassifier(model_name=cfg.classifier.model,
                         checkpoint_dir=resolve_path(cfg, 'models') / 'classifier',
                         device=select_models(cfg)['device'])
print('classifier model:', 'distilbert checkpoint' if clf.available else 'rules only')
analyzed = clf.analyze(sents)
decisions = extract_decisions(analyzed, cfg.classifier.decision_threshold)
actions = extract_actions(analyzed, m['participants'], cfg.classifier.action_threshold)
followups = extract_followups(analyzed)
print(f'decisions={len(decisions)} actions={len(actions)} followups={len(followups)}')

In [ ]:
print('[4/4] Results with provenance (auditable)')
print('DECISIONS')
for d in decisions:
    print(f'  [{d["confidence"]:.2f}] {d["speaker"]} @ {d["timestamp"]}: {d["text"]}')
print()
print('ACTIONS (action | owner | deadline)')
for a in actions:
    print(f'  [{a["confidence"]:.2f}] {a["action"]}')
    print(f'      owner={a["owner"]}  deadline={a["deadline"]}')
print()
print('FOLLOW-UPS')
for f in followups:
    print(f'  - {f["text"]}')